In [11]:
import os
from google.colab import userdata

# creating environment AND importing dT from Kaggle
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
import kagglehub
path = kagglehub.competition_download('titanic')
print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/titanic


In [12]:
import pandas as pd

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
#train.head()

In [13]:
train.shape
train.isnull().sum()
train.shape


(891, 12)

In [14]:
# Fill missing Age with the median age
train["Age"] = train["Age"].fillna(train["Age"].median())

# Fill missing Embarked with the most common value
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

# Drop columns we won't use
train = train.drop(columns=["Cabin", "Ticket", "Name", "PassengerId"])

# Check the result
train.isnull().sum()
train.shape
#train.head()

(891, 8)

In [15]:
# Sex: male -> 0, female -> 1


# Embarked: one yes/no column per port
train = pd.get_dummies(train, columns=["Embarked"], drop_first=True, dtype=int)
train["Sex"] = train["Sex"].map({"male": 0, "female": 1})

# Check the result
train.dtypes
train.shape
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,1
1,1,1,1,38.0,1,0,71.2833,0,0
2,1,3,1,26.0,0,0,7.9250,0,1
3,1,1,1,35.0,1,0,53.1000,0,1
4,0,3,0,35.0,0,0,8.0500,0,1


In [16]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = train.drop(columns=["Survived"])
y = train["Survived"]

# Split: 80% for training, 20% for the quiz
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_val.shape)

(712, 8) (179, 8)


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Step 5: train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 6: check accuracy on the quiz passengers
predictions = model.predict(X_val)
accuracy = accuracy_score(y_val, predictions)
print("Accuracy:", accuracy)

Accuracy: 0.7988826815642458


In [18]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

raw_train = pd.read_csv(os.path.join(path, "train.csv"))
raw_test = pd.read_csv(os.path.join(path, "test.csv"))

def prepare(df, ref):
    df = df.copy()
    df["Age"] = df["Age"].fillna(ref["Age"].median())
    df["Fare"] = df["Fare"].fillna(ref["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(ref["Embarked"].mode()[0])
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df = df.drop(columns=["Cabin", "Ticket", "Name", "PassengerId"])
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)
    return df

train_clean = prepare(raw_train, raw_train)
test_clean = prepare(raw_test, raw_train)

X = train_clean.drop(columns=["Survived"])
y = train_clean["Survived"]
test_clean = test_clean[X.columns]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)



Accuracy: 0.994413407821229


In [22]:
submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": model.predict(test_clean)
})
submission.to_csv("submission.csv", index=False)

print(test_clean.shape, test_clean.isnull().sum().sum())
print(submission.shape)
submission.head()

(418, 8) 0
(418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,1
4,896,0


In [23]:
from google.colab import files; files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>